[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/abpandey-dev/learn-python-from-scratch-to-advanced/blob/main/projects/03_subscription-churn.ipynb)

        *Industry Project 3 of 8 · Data Scientist · Intermediate · about 8–10 hours*

        # Predict Churn for a Subscription App

        **The situation.** *StreamBox*, a subscription video app, loses about one in five customers each period. The retention team can call customers and offer a discount, but the offer costs money, so they can't call everyone. They ask you for:

1. Who is likely to churn, with a score for every customer.
2. How many customers to call: where to set the cut-off.
3. *Why* customers churn, in terms the business can act on.
4. A list of the customers to call first.

        **What you'll hand in:** a tested churn model (baseline and a stronger model), a threshold chosen from the business cost, the top churn drivers, and a prioritised retention list with its expected value.

        **Skills:** EDA, train/test split, scikit-learn pipelines, logistic regression, gradient boosting, ROC AUC, cost-based thresholds, permutation importance

        **How this works:** each milestone has a task, a code cell for you to finish, and a ✅ check. Hints and a reference solution are hidden until you open them. Try for at least 20 minutes before opening the solution — the struggle is where the learning happens.

## The data

**churn_customers.csv** — one row per customer (5,000):

| Column | Meaning |
|---|---|
| customer_id | ID (not a feature!) |
| tenure_months | months as a customer |
| plan | basic / standard / premium |
| contract | monthly / annual |
| region | North / South / East / West |
| monthly_charge | ₹ per month |
| usage_hours | hours watched last month |
| support_tickets | tickets raised last quarter |
| payment_failures | failed payments last quarter |
| last_login_days | days since last login |
| churned | **target**: 1 if they left |

In [ ]:
# ▶ Run this cell first: it loads the libraries, the data and the checker.
import math, os, warnings
from pathlib import Path
import numpy as np, pandas as pd
warnings.filterwarnings("ignore")
RAW = "https://raw.githubusercontent.com/abpandey-dev/learn-python-from-scratch-to-advanced/main/projects/data/"


def load(name):
    """Read a project data file: from the local data/ folder if it is there, otherwise from GitHub."""
    local = Path("data") / name
    return pd.read_csv(local if local.exists() else RAW + name)


def _plain(x):
    if isinstance(x, pd.DataFrame):
        return {str(c): _plain(x[c].tolist()) for c in x.columns}
    if isinstance(x, pd.Series):
        return {_plain(k): _plain(v) for k, v in x.items()}
    if isinstance(x, np.ndarray):
        return _plain(x.tolist())
    if isinstance(x, np.generic):
        return _plain(x.item())
    if isinstance(x, (list, tuple)):
        return [_plain(v) for v in x]
    if isinstance(x, dict):
        return {_plain(k): _plain(v) for k, v in x.items()}
    return x


def _same(a, b, tol):
    if isinstance(b, bool) or isinstance(a, bool):
        return a == b
    if isinstance(b, (int, float)) and isinstance(a, (int, float)):
        return math.isclose(a, b, rel_tol=tol, abs_tol=tol if tol > 1e-3 else 1e-6)
    if isinstance(b, list) and isinstance(a, list):
        return len(a) == len(b) and all(_same(x, y, tol) for x, y in zip(a, b))
    if isinstance(b, dict) and isinstance(a, dict):
        return set(map(str, a)) == set(map(str, b)) and all(_same(a[k], b[k], tol) for k in a if k in b)
    return a == b


def _why(a, b):
    if a is None:
        return "your function returned None — finish the code and return the result."
    if type(a) != type(b) and not (isinstance(a, (int, float)) and isinstance(b, (int, float))):
        return f"expected a {type(b).__name__}, got a {type(a).__name__}."
    if isinstance(b, (list, dict)) and len(a) != len(b):
        return f"expected {len(b)} items, got {len(a)}."
    if isinstance(b, dict) and set(map(str, a)) != set(map(str, b)):
        return f"the keys should be {sorted(map(str, b))[:8]}."
    return "the values are not right yet. Re-read the task and the hints."


def check(name, fn, want, tol=1e-3):
    """Run your code for one milestone and compare it with the expected answer."""
    try:
        got = _plain(fn())
    except NotImplementedError:
        print(f"⬜ {name}: not started yet.")
        return False
    except Exception as e:
        print(f"❌ {name}: your code raised {type(e).__name__}: {e}")
        return False
    if _same(got, want, tol):
        print(f"✅ {name}: correct!")
        return True
    print(f"❌ {name}: not yet — {_why(got, want)}")
    return False

from sklearn.model_selection import train_test_split
from sklearn.compose import ColumnTransformer
from sklearn.preprocessing import OneHotEncoder, StandardScaler
from sklearn.pipeline import Pipeline
from sklearn.linear_model import LogisticRegression
from sklearn.ensemble import HistGradientBoostingClassifier
from sklearn.metrics import roc_auc_score
from sklearn.inspection import permutation_importance
df = load("churn_customers.csv")
CAT = ["plan", "contract", "region"]
NUM = ["tenure_months", "monthly_charge", "usage_hours", "support_tickets", "payment_failures", "last_login_days"]


# Helpers used by the checks (they call the functions you write).
def fitted(make_model):
    X_train, X_test, y_train, y_test = split(df)
    return make_model(X_train, y_train), X_test, y_test


def auc_of(make_model):
    model, X_test, y_test = fitted(make_model)
    return round(roc_auc_score(y_test, model.predict_proba(X_test)[:, 1]), 3)


def test_probs(make_model):
    model, X_test, y_test = fitted(make_model)
    return model.predict_proba(X_test)[:, 1], y_test.to_numpy()
print("Ready! Data loaded.")

## Milestone 1: Where is churn highest?

Write `churn_by(df, col)` returning `{category: churn rate %}` (rounded to 1) for a categorical column. The check runs it for `plan` and `contract`.

Then explore numeric columns yourself, e.g. `df.groupby('churned')[NUM].mean()`. Which ones look different for churners?

In [ ]:
def churn_by(df, col):
    raise NotImplementedError

In [ ]:
check("Where is churn highest?", lambda: [churn_by(df, 'plan'), churn_by(df, 'contract')], [{'basic': 27.9, 'premium': 21.5, 'standard': 22.9}, {'annual': 13.2, 'monthly': 32.3}], tol=0.001)

<details>
<summary><b>🔑 Reference solution (open after you've tried)</b></summary>

```python
def churn_by(df, col):
    return (df.groupby(col)["churned"].mean() * 100).round(1).to_dict()
```

</details>

## Milestone 2: Split without leaking

Write `split(df)` returning `X_train, X_test, y_train, y_test`:
- features = every column except `customer_id` and `churned`,
- target = `churned`,
- 25% test, **stratified** on the target, `random_state=42`.

The check looks at the sizes and the churn rate in each part (stratifying keeps them equal).

<details>
<summary><b>💡 Hint</b></summary>

`train_test_split(X, y, test_size=0.25, stratify=y, random_state=42)`.

</details>

In [ ]:
def split(df):
    raise NotImplementedError

In [ ]:
check("Split without leaking", lambda: [len(split(df)[0]), len(split(df)[1]), round(split(df)[2].mean(), 4), round(split(df)[3].mean(), 4)], [3750, 1250, 0.2485, 0.2488], tol=1e-09)

<details>
<summary><b>🔑 Reference solution (open after you've tried)</b></summary>

```python
def split(df):
    X = df.drop(columns=["customer_id", "churned"])
    y = df["churned"]
    return train_test_split(X, y, test_size=0.25, stratify=y, random_state=42)
```

</details>

## Milestone 3: A baseline you can explain

Write `baseline_model(X_train, y_train)` returning a **fitted** `Pipeline`:
- a `ColumnTransformer` that one-hot encodes `CAT` and standard-scales `NUM`,
- then `LogisticRegression(max_iter=1000)`.

The check reports test ROC AUC. Putting preprocessing inside the pipeline means it is learned from training data only, and the same steps run when the model is used later.

<details>
<summary><b>💡 Hint</b></summary>

`ColumnTransformer([('cat', OneHotEncoder(handle_unknown='ignore'), CAT), ('num', StandardScaler(), NUM)])`

</details>

In [ ]:
def baseline_model(X_train, y_train):
    raise NotImplementedError

In [ ]:
check("A baseline you can explain", lambda: auc_of(baseline_model), 0.761, tol=0.01)

<details>
<summary><b>🔑 Reference solution (open after you've tried)</b></summary>

```python
def baseline_model(X_train, y_train):
    prep = ColumnTransformer([("cat", OneHotEncoder(handle_unknown="ignore"), CAT), ("num", StandardScaler(), NUM)])
    return Pipeline([("prep", prep), ("model", LogisticRegression(max_iter=1000))]).fit(X_train, y_train)
```

</details>

## Milestone 4: A stronger model

Write `boosted_model(X_train, y_train)`: same preprocessing idea, but `HistGradientBoostingClassifier(max_iter=200, learning_rate=0.05, random_state=0)` (scaling isn't needed for trees; one-hot with `sparse_output=False`).

Compare the AUC with your baseline. Is the extra complexity worth it here? (On this data the honest answer may surprise you.)

In [ ]:
def boosted_model(X_train, y_train):
    raise NotImplementedError

In [ ]:
check("A stronger model", lambda: auc_of(boosted_model), 0.704, tol=0.015)

<details>
<summary><b>🔑 Reference solution (open after you've tried)</b></summary>

```python
def boosted_model(X_train, y_train):
    prep = ColumnTransformer([("cat", OneHotEncoder(handle_unknown="ignore", sparse_output=False), CAT)],
                             remainder="passthrough")
    model = HistGradientBoostingClassifier(max_iter=200, learning_rate=0.05, random_state=0)
    return Pipeline([("prep", prep), ("model", model)]).fit(X_train, y_train)
```

</details>

When the true relationship is close to linear (as here), a well-prepared logistic regression is as good as boosting and far easier to explain. Always compare against a simple baseline before reaching for complexity.

## Milestone 5: Choose the cut-off from the business cost

The retention team calls every customer whose churn probability is ≥ a threshold. Each call + discount costs **₹200**. A called customer who *would* have churned stays with probability **0.5**, and keeping a customer is worth **₹3,000**.

So for a threshold t, net value = Σ over called customers of (0.5 × 3000 × churned − 200).

Write `best_threshold(probs, y)` that tries t = 0.05, 0.10, …, 0.95 and returns `(best t, net value)`. The check uses the baseline model's test probabilities.

<details>
<summary><b>💡 Hint</b></summary>

For each t: `called = probs >= t`; value = `(0.5 * 3000 * y[called]).sum() - 200 * called.sum()`.

</details>

In [ ]:
def best_threshold(probs, y):
    raise NotImplementedError

In [ ]:
check("Choose the cut-off from the business cost", lambda: best_threshold(*test_probs(baseline_model)), [0.1, 251800.0], tol=0.03)

<details>
<summary><b>🔑 Reference solution (open after you've tried)</b></summary>

```python
def best_threshold(probs, y):
    best = None
    for t in np.round(np.arange(0.05, 0.96, 0.05), 2):
        called = probs >= t
        value = (0.5 * 3000 * y[called]).sum() - 200 * called.sum()
        if best is None or value > best[1]:
            best = (float(t), float(value))
    return best
```

</details>

Notice the best threshold is far below 0.5: a missed churner costs much more than a wasted call. The *default* 0.5 cut-off is almost never the business-optimal one.

## Milestone 6: What drives churn?

Write `top_drivers(model, X_test, y_test)` returning the names of the **3 most important** original columns by permutation importance (`scoring='roc_auc'`, `n_repeats=5`, `random_state=0`), most important first. The check uses the boosted model.

<details>
<summary><b>💡 Hint</b></summary>

`permutation_importance(model, X_test, y_test, scoring='roc_auc', n_repeats=5, random_state=0)` works on the whole pipeline, so it shuffles the original columns.

</details>

In [ ]:
def top_drivers(model, X_test, y_test):
    raise NotImplementedError

In [ ]:
check("What drives churn?", lambda: top_drivers(*fitted(boosted_model)), ['contract', 'last_login_days', 'tenure_months'], tol=1e-09)

<details>
<summary><b>🔑 Reference solution (open after you've tried)</b></summary>

```python
def top_drivers(model, X_test, y_test):
    r = permutation_importance(model, X_test, y_test, scoring="roc_auc", n_repeats=5, random_state=0)
    order = np.argsort(-r.importances_mean)[:3]
    return [X_test.columns[i] for i in order]
```

</details>

Turn drivers into actions: monthly contracts → offer an annual plan; payment failures → fix billing reminders; low usage / long since login → re-engagement emails.

## Milestone 7: The call list

The team can call **100** customers this week. Rank test customers by **expected value at risk** = churn probability × monthly_charge × 12, and take the top 100.

Write `call_list(model, X_test, y_test, n=100)` returning `(how many of the n actually churned, total expected value at risk rounded to 0)`. The check uses the baseline model.

In [ ]:
def call_list(model, X_test, y_test, n=100):
    raise NotImplementedError

In [ ]:
check("The call list", lambda: call_list(*fitted(baseline_model)), [56, 440433], tol=0.05)

<details>
<summary><b>🔑 Reference solution (open after you've tried)</b></summary>

```python
def call_list(model, X_test, y_test, n=100):
    p = model.predict_proba(X_test)[:, 1]
    value = p * X_test["monthly_charge"].to_numpy() * 12
    top = np.argsort(-value)[:n]
    return int(y_test.to_numpy()[top].sum()), round(float(value[top].sum()))
```

</details>

## Present your work

Write a short model report for the retention team and your manager:

1. **Problem and data**: what churn means here, the churn rate, the features.
2. **Models**: baseline vs boosted AUC, which you chose and why.
3. **Decision rule**: the threshold, how many customers it flags, and the expected net value vs calling no one or everyone.
4. **Drivers** and a recommended action for each.
5. **Call list**: the top 100, how many are expected to churn.
6. **Risks**: the model is correlational, it needs monitoring and retraining, and the offer's real effect should be measured with an A/B test.

## Put it on your resume

Write bullets with **your own numbers** from the milestones above. A template:

- Built a churn model for 5,000 subscribers with scikit-learn pipelines (logistic regression and gradient boosting, test AUC **0.XX**)
- Chose the decision threshold from offer cost and customer value, raising expected retention value to **₹X** vs the default cut-off
- Identified the top churn drivers with permutation importance and turned them into retention actions

Only claim what you built and measured. Put the notebook on your GitHub with a short README: problem, data, approach, results, what you'd do next.

## Interview questions about this project

Practise answering these out loud in 1–2 minutes each:

1. Why did you stratify the split, and why exclude customer_id?
2. Your boosted model wasn't much better than logistic regression. What does that tell you, and which would you deploy?
3. Why isn't 0.5 the right threshold? Walk me through your cost calculation.
4. Permutation importance says payment failures matter. Does that mean fixing payments will reduce churn?
5. How would you check that the retention offer actually works?
6. What would you monitor after deploying this model?

## Take it further

- Calibrate the probabilities (`CalibratedClassifierCV`) and check with a reliability curve.
- Tune the boosted model with cross-validated search and see whether AUC moves.
- Explain single predictions with SHAP values for the top 5 customers on the call list.
- Project 7 takes this model to production — do it next.